In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Khởi tạo SparkSession
spark = SparkSession.builder \
    .appName("PokemonBigDataAnalytics") \
    .getOrCreate()

print("Spark version:", spark.version)

# Đọc file CSV gốc từ Kaggle
df_pokemon = spark.read.csv(
    "pokemon_complete_stats.csv",
    header=True,
    inferSchema=True
)

# Xem 5 dòng đầu tiên và cấu trúc bảng
df_pokemon.show(5)
df_pokemon.printSchema()

Spark version: 4.0.4
+---+----------+---------+---------+---------------+------+------+---+------+-------+--------------+---------------+-----+--------------------+------------+------------+-----------+-----+---------+------------+
| id|      name|height_dm|weight_hg|base_experience|type_1|type_2| hp|attack|defense|special_attack|special_defense|speed|           abilities|  generation|is_legendary|is_mythical|color|    shape|capture_rate|
+---+----------+---------+---------+---------------+------+------+---+------+-------+--------------+---------------+-----+--------------------+------------+------------+-----------+-----+---------+------------+
|  1| bulbasaur|        7|       69|           64.0| grass|poison| 45|    49|     49|            65|             65|   45|overgrow, chlorop...|generation-i|       false|      false|green|quadruped|          45|
|  2|   ivysaur|       10|      130|          142.0| grass|poison| 60|    62|     63|            80|             80|   60|overgrow, chl

In [7]:
quality_check = df_pokemon.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.col("name").isNull().cast("int")).alias("missing_name"),
    F.sum(F.col("type_1").isNull().cast("int")).alias("missing_type1"),
    F.sum((F.col("hp") + F.col("attack") + F.col("defense") +
           F.col("special_attack") + F.col("special_defense") + F.col("speed") <= 0).cast("int")).alias("invalid_total_stats")
)

display(quality_check)

DataFrame[total_rows: bigint, missing_name: bigint, missing_type1: bigint, invalid_total_stats: bigint]

In [6]:
df_clean = (
    df_pokemon
    .filter(F.col("name").isNotNull())
    .filter(F.col("type_1").isNotNull())
    .filter(
        (F.col("hp") + F.col("attack") + F.col("defense") +
         F.col("special_attack") + F.col("special_defense") + F.col("speed")) > 0
    )
)

print(f"Số dòng ban đầu: {df_pokemon.count():,}")
print(f"Số dòng sau khi làm sạch: {df_clean.count():,}")

Số dòng ban đầu: 1,351
Số dòng sau khi làm sạch: 1,351


In [9]:
type_power_ranking = (
    df_clean
    .groupBy("type_1")
    .agg(
        F.count("name").alias("pokemon_count"),
        F.round(
            F.avg(F.col("hp") + F.col("attack") + F.col("defense") +
                  F.col("special_attack") + F.col("special_defense") + F.col("speed")), 2
        ).alias("avg_total_stats"),
        F.round(F.avg("attack"), 2).alias("avg_attack"),
        F.round(F.avg("defense"), 2).alias("avg_defense")
    )
    .orderBy(F.col("avg_total_stats").desc())
)

type_power_ranking.show(10, truncate=False)

+--------+-------------+---------------+----------+-----------+
|type_1  |pokemon_count|avg_total_stats|avg_attack|avg_defense|
+--------+-------------+---------------+----------+-----------+
|dragon  |59           |544.73         |103.24    |83.98      |
|steel   |52           |504.71         |96.46     |113.92     |
|psychic |86           |489.33         |74.87     |72.35      |
|fighting|62           |486.24         |110.44    |82.08      |
|fire    |84           |468.02         |86.19     |70.73      |
|fairy   |35           |463.03         |71.23     |74.43      |
|dark    |63           |462.57         |88.95     |73.46      |
|flying  |13           |457.69         |82.77     |68.38      |
|rock    |83           |456.82         |89.59     |95.6       |
|ground  |50           |449.46         |97.06     |88.2       |
+--------+-------------+---------------+----------+-----------+
only showing top 10 rows


In [10]:
# Ghi dữ liệu xuống định dạng Parquet
output_path = "pokemon_clean_data.parquet"
df_clean.write.mode("overwrite").parquet(output_path)

print(f"Đã lưu thành công dữ liệu sạch vào định dạng Parquet tại: {output_path}")

Đã lưu thành công dữ liệu sạch vào định dạng Parquet tại: pokemon_clean_data.parquet


In [11]:
# Chuyển đổi DataFrame PySpark sang Pandas để xuất file CSV gọn nhẹ cho Streamlit
df_clean_pd = df_clean.toPandas()

# Lưu thành file CSV sạch
df_clean_pd.to_csv("pokemon_clean.csv", index=False)

print("Đã xuất thành công file pokemon_clean.csv để làm nguồn dữ liệu cho app!")

Đã xuất thành công file pokemon_clean.csv để làm nguồn dữ liệu cho app!
